In [0]:
# =============================================================
# 10_dlt_pipeline — Delta Live Tables
# Author: oakville3456
# Branch: feature/priority9-dlt
# Purpose: Declare Bronze → Silver → Gold as DLT tables
# =============================================================

import dlt
from pyspark.sql import functions as F

RAW = "abfss://raw-landing@saretailsalesdev.dfs.core.windows.net/sales/"

# ── Bronze ──────────────────────────────────────────────────
@dlt.table(
    name    = "bronze_sales",
    comment = "Raw sales data ingested from ADLS raw-landing via Auto Loader"
)
def bronze_sales():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(RAW)
        .withColumn("_ingested_at", F.current_timestamp())
    )

# ── Silver ──────────────────────────────────────────────────
@dlt.table(
    name    = "silver_sales",
    comment = "Cleaned and deduplicated sales data"
)
@dlt.expect("valid_order_id",  "order_id IS NOT NULL")
@dlt.expect("valid_price",     "price > 0")
@dlt.expect_or_drop("valid_order_id_not_null", "order_id IS NOT NULL")
def silver_sales():
    return (
        dlt.read_stream("bronze_sales")
        .filter(F.col("order_id").isNotNull())
        .filter(F.col("price") > 0)
        .withColumn("order_date", F.try_to_date("order_date"))
        .withColumn("revenue",    F.col("quantity") * F.col("price"))
    )

# ── Gold ────────────────────────────────────────────────────
@dlt.table(
    name    = "gold_sales_daily",
    comment = "Daily revenue aggregated by store"
)
def gold_sales_daily():
    return (
        dlt.read("silver_sales")
        .filter(F.col("order_date").isNotNull())
        .groupBy("store_id", "order_date")
        .agg(
            F.sum("revenue").alias("total_revenue"),
            F.count("order_id").alias("order_count"),
            F.approx_count_distinct("customer_id").alias("unique_customers")
        )
    )